# Task 3: Data Cleaning and Data Quality Improvement

## Project Objective

The objective of this project is to demonstrate professional-level data cleaning skills by transforming a deliberately messy employee dataset into a clean and analysis-ready dataset.

The project follows a systematic data cleaning process that includes:

- Initial data inspection
- Data quality assessment
- Missing data handling
- Duplicate detection and removal
- Data standardisation
- Value range validation
- Outlier detection using the IQR method
- Data type correction
- Before-and-after data quality comparison
- Exporting the cleaned dataset

Every major data cleaning decision is documented and justified throughout the notebook.

## Technologies Used

- Python
- Pandas
- NumPy
- Jupyter Notebook

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load Dataset
df = pd.read_csv("Messy_Employee_dataset.csv")

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [3]:
# Create Original Dataset Copy

df_original = df.copy()

print("Original dataset copy created successfully.")

Original dataset copy created successfully.


In [4]:
# Display Dataset Shape

print("Dataset Shape")
print("-" * 40)

print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Dataset Shape
----------------------------------------
Number of rows: 1020
Number of columns: 12


In [5]:
# Display First Five Rows

df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [6]:
# Display Column Names

print("Dataset Columns:")
print(df.columns.tolist())

Dataset Columns:
['Employee_ID', 'First_Name', 'Last_Name', 'Age', 'Department_Region', 'Status', 'Join_Date', 'Salary', 'Email', 'Phone', 'Performance_Score', 'Remote_Work']


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   str    
 1   First_Name         1020 non-null   str    
 2   Last_Name          1020 non-null   str    
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   str    
 5   Status             1020 non-null   str    
 6   Join_Date          1020 non-null   str    
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   str    
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   str    
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), str(8)
memory usage: 165.7 KB


In [8]:
# Inspect Data Types
print("Current Data Types:")
print("-" * 40)

df.dtypes

Current Data Types:
----------------------------------------


Employee_ID              str
First_Name               str
Last_Name                str
Age                  float64
Department_Region        str
Status                   str
Join_Date                str
Salary               float64
Email                    str
Phone                  int64
Performance_Score        str
Remote_Work             bool
dtype: object

## INITIAL DATA QUALITY REPORT

In [9]:
# Missing Values Per Column
missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing Values": [
        df[column].isna().sum()
        for column in df.columns
    ],
    "Missing Percentage": [
        round(df[column].isna().sum() / len(df) * 100, 2)
        for column in df.columns
    ]
})

missing_report

,Column,Missing Values,Missing Percentage
0,Employee_ID,0,0.00
1,First_Name,0,0.00
2,Last_Name,0,0.00
3,Age,211,20.69
4,Department_Region,0,0.00
5,Status,0,0.00
6,Join_Date,0,0.00
7,Salary,24,2.35
8,Email,0,0.00
9,Phone,0,0.00


In [10]:
# check duplicate rows
duplicate_rows_before = df.duplicated().sum()

print(
    f"Number of exact duplicate rows: "
    f"{duplicate_rows_before}"
)

Number of exact duplicate rows: 0


In [11]:
# Define Expected Data Types
expected_dtypes = {
    "Employee_ID": "string",
    "First_Name": "string",
    "Last_Name": "string",
    "Age": "float",
    "Department_Region": "string",
    "Status": "string",
    "Join_Date": "datetime64[ns]",
    "Salary": "float",
    "Email": "string",
    "Phone": "string",
    "Performance_Score": "string",
    "Remote_Work": "string"
}

In [12]:
# Data Type Quality Report
dtype_report = pd.DataFrame({
    "Column": df.columns,
    "Current Dtype": [
        str(df[column].dtype)
        for column in df.columns
    ],
    "Expected Dtype": [
        expected_dtypes.get(
            column,
            "Not Defined"
        )
        for column in df.columns
    ]
})

dtype_report["Dtype Issue"] = (
    dtype_report["Current Dtype"] !=
    dtype_report["Expected Dtype"]
)

dtype_report

,Column,Current Dtype,Expected Dtype,Dtype Issue
0,Employee_ID,str,string,True
1,First_Name,str,string,True
2,Last_Name,str,string,True
3,Age,float64,float,True
4,Department_Region,str,string,True
5,Status,str,string,True
6,Join_Date,str,datetime64[ns],True
7,Salary,float64,float,True
8,Email,str,string,True
9,Phone,int64,string,True


In [13]:
# Numerical Value Range Analysis

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical Columns:")
print(numeric_columns)

range_report = pd.DataFrame({
    "Column": numeric_columns,
    "Minimum": [
        df[column].min()
        for column in numeric_columns
    ],
    "Maximum": [
        df[column].max()
        for column in numeric_columns
    ],
    "Mean": [
        df[column].mean()
        for column in numeric_columns
    ],
    "Median": [
        df[column].median()
        for column in numeric_columns
    ]
})

range_report

Numerical Columns:
['Age', 'Salary', 'Phone']


,Column,Minimum,Maximum,Mean,Median
0,Age,2.500000e+01,40.00,3.248455e+01,3.000000e+01
1,Salary,5.004732e+04,119971.65,8.515506e+04,8.554787e+04
2,Phone,-9.994973e+09,-3896086.00,-4.942253e+09,-4.943997e+09


In [14]:
# Check Specific Value Range Anomalies

print("Age Validation")
print("-" * 40)

invalid_age = df[
    (df["Age"] < 18) |
    (df["Age"] > 100)
]

print(
    f"Invalid Age Values: "
    f"{len(invalid_age)}"
)


print("\nSalary Validation")
print("-" * 40)

invalid_salary = df[
    df["Salary"] < 0
]

print(
    f"Negative Salary Values: "
    f"{len(invalid_salary)}"
)


print("\nPhone Validation")
print("-" * 40)

negative_phone_count = (
    df["Phone"] < 0
).sum()

print(
    f"Negative Phone Values: "
    f"{negative_phone_count}"
)

Age Validation
----------------------------------------
Invalid Age Values: 0

Salary Validation
----------------------------------------
Negative Salary Values: 0

Phone Validation
----------------------------------------
Negative Phone Values: 1020


In [15]:
# Complete Initial Data Quality Report

data_quality_report = pd.DataFrame({
    "Column": df.columns,
    
    "Data Type": [
        str(df[column].dtype)
        for column in df.columns
    ],
    
    "Missing Values": [
        df[column].isna().sum()
        for column in df.columns
    ],
    
    "Missing Percentage": [
        round(
            df[column].isna().sum()
            / len(df) * 100,
            2
        )
        for column in df.columns
    ],
    
    "Unique Values": [
        df[column].nunique()
        for column in df.columns
    ]
})

data_quality_report

,Column,Data Type,Missing Values,Missing Percentage,Unique Values
0,Employee_ID,str,0,0.00,1020
1,First_Name,str,0,0.00,8
2,Last_Name,str,0,0.00,8
3,Age,float64,211,20.69,4
4,Department_Region,str,0,0.00,36
5,Status,str,0,0.00,3
6,Join_Date,str,0,0.00,760
7,Salary,float64,24,2.35,978
8,Email,str,0,0.00,64
9,Phone,int64,0,0.00,1020


# Initial Data Quality Assessment

An initial assessment was conducted to identify potential data quality issues before performing any cleaning operations.

The assessment examined:

- Missing values in every column
- Duplicate rows
- Current data types
- Expected data types
- Numerical value ranges
- Potential invalid values

The dataset contains missing values in numerical variables, including `Age` and `Salary`. These values require an appropriate imputation strategy to preserve useful employee records.

The dataset was also examined for duplicate records. The number of duplicates identified and removed is documented later in the cleaning process.

Several data type issues were identified. Employee IDs and telephone numbers are identifiers and should therefore be represented as strings rather than numerical values. The `Join_Date` column also requires conversion to a datetime format.

Additional value validation was performed to identify impossible values, such as invalid ages, negative salaries, and negative phone numbers.

## MISSING DATA HANDLING

In [16]:
# Missing Values Before Cleaning

missing_before = df.isna().sum()

print("Missing Values Before Cleaning:")
print(missing_before)

Missing Values Before Cleaning:
Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64


In [17]:
# Record Missing Values Before Imputation
age_missing_before = df["Age"].isna().sum()

salary_missing_before = df["Salary"].isna().sum()

print(
    f"Missing Age values: "
    f"{age_missing_before}"
)

print(
    f"Missing Salary values: "
    f"{salary_missing_before}"
)

Missing Age values: 211
Missing Salary values: 24


In [18]:
# Examine Age Distribution
df["Age"].describe()

count    809.000000
mean      32.484549
std        5.656860
min       25.000000
25%       25.000000
50%       30.000000
75%       40.000000
max       40.000000
Name: Age, dtype: float64

In [19]:
# Impute Missing Age Values Using Median

age_median = df["Age"].median()

df["Age"] = df["Age"].fillna(
    age_median
)

print(
    "Missing Age values after imputation:",
    df["Age"].isna().sum()
)

Missing Age values after imputation: 0


In [20]:
# Examine Salary Distribution

df["Salary"].describe()

count       996.000000
mean      85155.056396
std       19873.727918
min       50047.320000
25%       68392.487500
50%       85547.870000
75%      100974.027500
max      119971.650000
Name: Salary, dtype: float64

In [21]:
# Impute Missing Salary Values Using Median

salary_median = df["Salary"].median()

df["Salary"] = df["Salary"].fillna(
    salary_median
)

print(
    "Missing Salary values after imputation:",
    df["Salary"].isna().sum()
)

Missing Salary values after imputation: 0


In [22]:
# Verify Missing Values After Imputation

print("Missing Values After Numerical Imputation:")
print("-" * 50)

df.isna().sum()

Missing Values After Numerical Imputation:
--------------------------------------------------


Employee_ID          0
First_Name           0
Last_Name            0
Age                  0
Department_Region    0
Status               0
Join_Date            0
Salary               0
Email                0
Phone                0
Performance_Score    0
Remote_Work          0
dtype: int64

# Missing Data Handling Strategy

## Age

The `Age` column contained missing values. Median imputation was selected to replace these missing values.

The median was considered appropriate because age is a numerical variable and the median is less sensitive to unusually high or low observations than the mean. This method also preserves employee records that would otherwise be lost through row deletion.

## Salary

The `Salary` column also contained missing values. Median imputation was used for this variable.

Salary distributions can be influenced by differences in job roles and employee characteristics. The median provides a robust estimate of the central value and reduces the potential influence of extreme salary observations.

## Other Missing Values

Other columns were checked for missing values during the initial data quality assessment. Any additional missing values created during later cleaning stages are evaluated separately and handled according to their importance and the reliability of possible imputation methods.

## DUPLICATE DETECTION AND REMOVAL

In [23]:
# Check Duplicate Rows

duplicate_rows_before = df.duplicated().sum()

print(
    f"Duplicate rows before cleaning: "
    f"{duplicate_rows_before}"
)

Duplicate rows before cleaning: 0


In [24]:
# Remove Exact Duplicate Rows

rows_before_duplicate_removal = len(df)

df = df.drop_duplicates()

rows_after_duplicate_removal = len(df)

duplicates_removed = (
    rows_before_duplicate_removal -
    rows_after_duplicate_removal
)

print(
    f"Duplicate rows removed: "
    f"{duplicates_removed}"
)

print(
    f"Rows remaining: "
    f"{len(df)}"
)

Duplicate rows removed: 0
Rows remaining: 1020


In [25]:
# Check Duplicate Employee IDs

duplicate_employee_ids = (
    df["Employee_ID"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate Employee IDs: "
    f"{duplicate_employee_ids}"
)

Duplicate Employee IDs: 0


# Duplicate Detection and Removal

The dataset was examined for exact duplicate rows before any records were removed.

The `drop_duplicates()` method was applied to ensure that any exact duplicate records were removed from the dataset. The number of removed records was calculated by comparing the dataset row count before and after the cleaning operation.

Employee IDs were also checked separately for duplication because an employee identifier should normally be unique.

If no duplicate rows were identified, zero records were removed. This result is still documented because duplicate detection is an important component of a professional data cleaning process.

## DATA STANDARDISATION

In [26]:
# Inspect Categorical Values

categorical_columns = [
    "Department_Region",
    "Status",
    "Performance_Score",
    "Remote_Work"
]

for column in categorical_columns:

    print("\n" + "=" * 50)
    print(f"Column: {column}")
    print("=" * 50)

    print(
        df[column]
        .value_counts(
            dropna=False
        )
    )


Column: Department_Region
Department_Region
HR-Florida               41
DevOps-California        35
Sales-Nevada             35
Admin-Nevada             34
Admin-California         34
DevOps-Florida           34
DevOps-New York          33
HR-New York              33
Sales-Florida            33
DevOps-Illinois          33
Finance-Illinois         33
Finance-California       32
Sales-California         31
Sales-Illinois           30
Finance-Texas            29
Admin-Illinois           29
Finance-Nevada           29
Cloud Tech-Texas         29
Cloud Tech-California    29
Cloud Tech-Florida       28
DevOps-Texas             27
DevOps-Nevada            27
Sales-New York           26
HR-California            26
Admin-Florida            25
Cloud Tech-New York      24
Finance-Florida          24
HR-Nevada                24
HR-Illinois              24
Sales-Texas              23
HR-Texas                 23
Finance-New York         23
Admin-Texas              22
Admin-New York           22
Clo

In [27]:
# Standardise Text-Based Columns

text_columns = [
    "First_Name",
    "Last_Name",
    "Department_Region",
    "Status",
    "Performance_Score",
    "Remote_Work"
]

for column in text_columns:

    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
    )

print(
    "Leading and trailing spaces removed."
)

Leading and trailing spaces removed.


In [28]:
# Standardise Names
df["First_Name"] = (
    df["First_Name"]
    .str.title()
)

df["Last_Name"] = (
    df["Last_Name"]
    .str.title()
)

In [29]:
# Standardise Status

df["Status"] = (
    df["Status"]
    .str.title()
)

In [30]:
# Standardise Performance Score
df["Performance_Score"] = (
    df["Performance_Score"]
    .str.title()
)

In [31]:
# Standardise Remote Work Values

df["Remote_Work"] = (
    df["Remote_Work"]
    .str.strip()
    .str.title()
)

In [32]:
# Standardise Email Addresses

df["Email"] = (
    df["Email"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [33]:
print("Status:")
print(df["Status"].value_counts(dropna=False))

print("\nPerformance Score:")
print(df["Performance_Score"].value_counts(dropna=False))

print("\nRemote Work:")
print(df["Remote_Work"].value_counts(dropna=False))

Status:
Status
Pending     356
Active      352
Inactive    312
Name: count, dtype: int64[pyarrow]

Performance Score:
Performance_Score
Good         270
Average      267
Excellent    267
Poor         216
Name: count, dtype: int64[pyarrow]

Remote Work:
Remote_Work
True     513
False    507
Name: count, dtype: int64[pyarrow]


In [34]:
# Inspect Department Region Format

df["Department_Region"].value_counts().head(20)

Department_Region
HR-Florida               41
DevOps-California        35
Sales-Nevada             35
Admin-Nevada             34
Admin-California         34
DevOps-Florida           34
DevOps-New York          33
HR-New York              33
Sales-Florida            33
DevOps-Illinois          33
Finance-Illinois         33
Finance-California       32
Sales-California         31
Sales-Illinois           30
Finance-Texas            29
Admin-Illinois           29
Finance-Nevada           29
Cloud Tech-Texas         29
Cloud Tech-California    29
Cloud Tech-Florida       28
Name: count, dtype: int64[pyarrow]

In [35]:
# Split Department and Region
split_department = (
    df["Department_Region"]
    .str.split(
        "-",
        n=1,
        expand=True
    )
)

df["Department"] = (
    split_department[0]
    .str.strip()
    .str.title()
)

df["Region"] = (
    split_department[1]
    .str.strip()
    .str.title()
)

In [36]:
df[
    [
        "Department_Region",
        "Department",
        "Region"
    ]
].head()

,Department_Region,Department,Region
0,DevOps-California,Devops,California
1,Finance-Texas,Finance,Texas
2,Admin-Nevada,Admin,Nevada
3,Admin-Nevada,Admin,Nevada
4,Cloud Tech-Florida,Cloud Tech,Florida


# Data Standardisation

Text-based variables were standardised to improve consistency and prepare the dataset for future analysis.

Leading and trailing spaces were removed from text fields. Employee names were converted to title case, while email addresses were converted to lowercase to create a consistent format.

Categorical variables such as employee status, performance score, and remote work status were reviewed and standardised based on the values present in the dataset.

The `Department_Region` column was also examined to determine whether department and regional information could be separated. Where the formatting was consistent, separate `Department` and `Region` columns were created to improve the analytical usefulness of the dataset.

These changes improve consistency without altering the underlying meaning of the employee information.

## DATA TYPE CORRECTION

In [37]:
df["Employee_ID"] = (
    df["Employee_ID"]
    .astype("string")
)

In [38]:
# Convert Names and Text Fields to String

string_columns = [
    "First_Name",
    "Last_Name",
    "Department_Region",
    "Status",
    "Email",
    "Performance_Score",
    "Remote_Work"
]

for column in string_columns:

    df[column] = (
        df[column]
        .astype("string")
    )

print("Text columns converted to string.")

Text columns converted to string.


## DATE STANDARDISATION

In [39]:
# Inspect Original Date Values
print("Sample Join_Date values:")

df["Join_Date"].head(15)

Sample Join_Date values:


0       4/2/2021
1      7/10/2020
2      12/7/2023
3     11/27/2021
4       1/5/2022
5      6/10/2020
6       4/3/2020
7      7/17/2022
8      12/8/2023
9       8/4/2022
10      7/6/2024
11      9/4/2023
12      3/5/2024
13    10/10/2020
14     6/10/2024
Name: Join_Date, dtype: str

In [40]:
# Convert Join Date to Datetime
df["Join_Date"] = pd.to_datetime(
    df["Join_Date"],
    errors="coerce",
    format="mixed"
)

print(
    "Join_Date converted to datetime."
)

Join_Date converted to datetime.


In [41]:
# Check Invalid or Missing Dates
invalid_date_count = (
    df["Join_Date"]
    .isna()
    .sum()
)

print(
    f"Missing or invalid Join_Date values: "
    f"{invalid_date_count}"
)

Missing or invalid Join_Date values: 0


# Join Date Standardisation

The `Join_Date` column was originally stored as a text-based variable. It was converted to datetime format to ensure that dates can be correctly interpreted and used in future chronological analysis.

The conversion process used error handling to identify invalid or unparseable date values.

Where all dates were successfully converted, no additional treatment was necessary. If invalid dates were identified, the affected records were reviewed before deciding whether they should be removed or retained.

## PHONE NUMBER VALIDATION

In [42]:
# Convert Phone to String
df["Phone"] = (
    df["Phone"]
    .astype("string")
    .str.strip()
)

In [43]:
# Identify Negative Phone Numbers

invalid_phone_mask = (
    df["Phone"]
    .str.startswith(
        "-",
        na=False
    )
)

invalid_phone_count = (
    invalid_phone_mask.sum()
)

print(
    f"Invalid negative phone numbers: "
    f"{invalid_phone_count}"
)

Invalid negative phone numbers: 1020


In [44]:
# Mark Invalid Phone Numbers as Missing

df.loc[
    invalid_phone_mask,
    "Phone"
] = pd.NA

print(
    "Invalid phone numbers have been "
    "converted to missing values."
)

Invalid phone numbers have been converted to missing values.


In [45]:
# Decide How to Handle Missing Phone Numbers

print(
    f"Missing Phone values after validation: "
    f"{df['Phone'].isna().sum()}"
)

Missing Phone values after validation: 1020


In [46]:
print(df_original["Phone"].head(10))
print("\nOriginal Phone Data Type:")
print(df_original["Phone"].dtype)

0   -1651623197
1   -1898471390
2   -5596363211
3   -3476490784
4   -1586734256
5   -5409003485
6   -4518376063
7   -4134327559
8   -4177656123
9   -8156985699
Name: Phone, dtype: int64

Original Phone Data Type:
int64


In [47]:
print("Original Phone Missing Values:")
print(df_original["Phone"].isna().sum())

Original Phone Missing Values:
0


In [48]:
df["Phone"] = df_original["Phone"].copy()

In [49]:
print("Phone Missing Values After Restoration:")
print(df["Phone"].isna().sum())

Phone Missing Values After Restoration:
0


In [50]:
df["Phone"] = (
    df["Phone"]
    .astype("string")
    .str.strip()
)

In [51]:
print(df["Phone"].head(10))

0    -1651623197
1    -1898471390
2    -5596363211
3    -3476490784
4    -1586734256
5    -5409003485
6    -4518376063
7    -4134327559
8    -4177656123
9    -8156985699
Name: Phone, dtype: string


In [52]:
df["Region"] = (
    df["Region"]
    .astype("string")
    .str.strip()
    .str.title()
)

In [53]:
print(df.dtypes)

Employee_ID                  string
First_Name                   string
Last_Name                    string
Age                         float64
Department_Region            string
Status                       string
Join_Date            datetime64[us]
Salary                      float64
Email                        string
Phone                        string
Performance_Score            string
Remote_Work                  string
Department                   string
Region                       string
dtype: object


In [54]:
# Final missing values
final_missing_values = df.isna().sum()

print("Final Missing Values:")
print(final_missing_values)

# Final duplicates
final_duplicate_count = df.duplicated().sum()

print("\nFinal Duplicate Rows:")
print(final_duplicate_count)

# Final shape
print("\nFinal Dataset Shape:")
print(df.shape)

# Final data types
print("\nFinal Data Types:")
print(df.dtypes)

Final Missing Values:
Employee_ID          0
First_Name           0
Last_Name            0
Age                  0
Department_Region    0
Status               0
Join_Date            0
Salary               0
Email                0
Phone                0
Performance_Score    0
Remote_Work          0
Department           0
Region               0
dtype: int64

Final Duplicate Rows:
0

Final Dataset Shape:
(1020, 14)

Final Data Types:
Employee_ID                  string
First_Name                   string
Last_Name                    string
Age                         float64
Department_Region            string
Status                       string
Join_Date            datetime64[us]
Salary                      float64
Email                        string
Phone                        string
Performance_Score            string
Remote_Work                  string
Department                   string
Region                       string
dtype: object


In [55]:
df["Phone"] = df_original["Phone"].copy()

In [56]:
# Correct negative phone number formatting

df["Phone"] = (
    df_original["Phone"]
    .astype("string")
    .str.replace("-", "", regex=False)
    .str.strip()
)

print("Sample cleaned phone numbers:")
print(df["Phone"].head(10))

Sample cleaned phone numbers:
0    1651623197
1    1898471390
2    5596363211
3    3476490784
4    1586734256
5    5409003485
6    4518376063
7    4134327559
8    4177656123
9    8156985699
Name: Phone, dtype: string


In [57]:
# Validate phone number length

phone_length_check = (
    df["Phone"]
    .str.len()
    .value_counts()
    .sort_index()
)

print("Phone number length distribution:")
print(phone_length_check)

Phone number length distribution:
Phone
7       2
8      12
9      78
10    928
Name: count, dtype: Int64


In [58]:
# Check for invalid phone numbers

invalid_phone_numbers = df[
    ~df["Phone"].str.fullmatch(r"\d{10}", na=False)
]

print(
    "Invalid phone numbers after cleaning:",
    len(invalid_phone_numbers)
)

Invalid phone numbers after cleaning: 92


# Phone Number Cleaning Decision

The original `Phone` column was stored as an integer data type, and all records contained negative values. Telephone numbers are identifiers rather than numerical measurements; therefore, negative signs have no analytical meaning.

The negative sign was removed and the values were converted to the string data type. The resulting values were subsequently validated to confirm that they contained the expected numerical characters and length.

The `Phone` column was retained as a string to preserve its role as an identifier and to prevent unintended mathematical operations.

# Phone Number Validation

The phone number column was originally represented as a numerical variable, despite telephone numbers being identifiers rather than quantities.

The column was converted to a string data type.

Negative phone numbers were identified as invalid. Rather than simply removing the negative sign and assuming the remaining digits were correct, these values were converted to missing values.

Missing phone numbers were retained because the absence of contact information does not necessarily make the employee record unusable for analytical purposes. Additionally, telephone numbers cannot be reliably imputed using statistical methods.

## FINAL NUMERICAL DATA TYPE CORRECTION

In [59]:
# Convert Age and Salary to Numeric
df["Age"] = pd.to_numeric(
    df["Age"],
    errors="coerce"
)

df["Salary"] = pd.to_numeric(
    df["Salary"],
    errors="coerce"
)

In [60]:
# Convert Age and Salary to Float

df["Age"] = df["Age"].astype(float)

df["Salary"] = df["Salary"].astype(float)

print(
    "Age and Salary converted to float."
)

Age and Salary converted to float.


## OUTLIER DETECTION USING THE IQR METHOD

In [61]:
# Define IQR Outlier Function

def detect_outliers_iqr(data, column):

    Q1 = data[column].quantile(0.25)

    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR

    upper_bound = Q3 + 1.5 * IQR

    outlier_mask = (
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    )

    outliers = data[
        outlier_mask
    ]

    return {
        "Column": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": len(outliers)
    }

In [62]:
# Detect Age Outliers

age_outlier_results = (
    detect_outliers_iqr(
        df,
        "Age"
    )
)

age_outlier_results

{'Column': 'Age',
 'Q1': np.float64(30.0),
 'Q3': np.float64(35.0),
 'IQR': np.float64(5.0),
 'Lower Bound': np.float64(22.5),
 'Upper Bound': np.float64(42.5),
 'Outlier Count': 0}

In [63]:
# Detect Salary Outliers
salary_outlier_results = (
    detect_outliers_iqr(
        df,
        "Salary"
    )
)

salary_outlier_results

{'Column': 'Salary',
 'Q1': np.float64(68811.2325),
 'Q3': np.float64(100372.6625),
 'IQR': np.float64(31561.430000000008),
 'Lower Bound': np.float64(21469.087499999987),
 'Upper Bound': np.float64(147714.80750000002),
 'Outlier Count': 0}

In [64]:
# Create Complete Outlier Summary

outlier_summary = pd.DataFrame([
    
    detect_outliers_iqr(
        df,
        "Age"
    ),
    
    detect_outliers_iqr(
        df,
        "Salary"
    )
])

outlier_summary

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,Age,30.0000,35.0000,5.00,22.5000,42.5000,0
1,Salary,68811.2325,100372.6625,31561.43,21469.0875,147714.8075,0


In [65]:
# Display Actual Outliers

def get_outliers_iqr(data, column):

    Q1 = data[column].quantile(0.25)

    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR

    upper_bound = Q3 + 1.5 * IQR

    return data[
        (data[column] < lower_bound) |
        (data[column] > upper_bound)
    ]

In [66]:
# Age outliers

age_outliers = get_outliers_iqr(
    df,
    "Age"
)

age_outliers

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work,Department,Region


In [67]:
# Salary outliers
salary_outliers = get_outliers_iqr(
    df,
    "Salary"
)

salary_outliers

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work,Department,Region


In [68]:
# Outlier Treatment Decision

print(
    "Age Outliers:",
    len(age_outliers)
)

print(
    "Salary Outliers:",
    len(salary_outliers)
)

Age Outliers: 0
Salary Outliers: 0


# Outlier Detection and Treatment

The Interquartile Range (IQR) method was applied to the numerical variables `Age` and `Salary` to identify potential outliers.

The analysis identified zero outliers in both the `Age` and `Salary` columns. Therefore, no observations required removal or capping.

This decision was based on the calculated IQR boundaries and ensures that the original dataset is preserved without unnecessarily modifying valid employee records.

## FINAL DATA TYPE VERIFICATION

In [69]:
# Check Final Data Types

print("Final Data Types:")
print("-" * 50)

df.dtypes

Final Data Types:
--------------------------------------------------


Employee_ID                  string
First_Name                   string
Last_Name                    string
Age                         float64
Department_Region            string
Status                       string
Join_Date            datetime64[us]
Salary                      float64
Email                        string
Phone                        string
Performance_Score            string
Remote_Work                  string
Department                   string
Region                       string
dtype: object

In [70]:
# Final Missing Values

final_missing_values = df.isna().sum()

print("Final Missing Values:")
print(final_missing_values)

Final Missing Values:
Employee_ID          0
First_Name           0
Last_Name            0
Age                  0
Department_Region    0
Status               0
Join_Date            0
Salary               0
Email                0
Phone                0
Performance_Score    0
Remote_Work          0
Department           0
Region               0
dtype: int64


In [71]:
# Final Duplicate Count

final_duplicate_count = (
    df.duplicated().sum()
)

print(
    f"Final duplicate rows: "
    f"{final_duplicate_count}"
)

Final duplicate rows: 0


In [72]:
# Final Dataset Shape

print(
    f"Final dataset shape: "
    f"{df.shape}"
)

Final dataset shape: (1020, 14)


## DTYPE ACCURACY

In [73]:
# Update Expected Data Types

final_expected_dtypes = {
    "Employee_ID": "string",
    "First_Name": "string",
    "Last_Name": "string",
    "Age": "float",
    "Department_Region": "string",
    "Status": "string",
    "Join_Date": "datetime64[ns]",
    "Salary": "float",
    "Email": "string",
    "Phone": "string",
    "Performance_Score": "string",
    "Remote_Work": "string"
}

if "Department" in df.columns:

    final_expected_dtypes[
        "Department"
    ] = "string"

if "Region" in df.columns:

    final_expected_dtypes[
        "Region"
    ] = "string"

In [74]:
# Create Dtype Accuracy Function

def calculate_dtype_accuracy(
    dataframe,
    expected_types
):

    correct_columns = 0

    total_columns = len(
        expected_types
    )

    for column, expected_type in expected_types.items():

        if column in dataframe.columns:

            if (
                str(dataframe[column].dtype)
                == expected_type
            ):

                correct_columns += 1

    return round(
        correct_columns
        / total_columns
        * 100,
        2
    )

In [75]:
# Define expected data type categories

expected_type_categories = {
    "Employee_ID": "string",
    "First_Name": "string",
    "Last_Name": "string",
    "Age": "numeric",
    "Department_Region": "string",
    "Status": "string",
    "Join_Date": "datetime",
    "Salary": "numeric",
    "Email": "string",
    "Phone": "string",
    "Performance_Score": "string",
    "Remote_Work": "string",
    "Department": "string",
    "Region": "string"
}

In [76]:
# Function to calculate data type accuracy

def check_dtype_category(series, expected_category):

    if expected_category == "string":
        return pd.api.types.is_string_dtype(series)

    elif expected_category == "numeric":
        return pd.api.types.is_numeric_dtype(series)

    elif expected_category == "datetime":
        return pd.api.types.is_datetime64_any_dtype(series)

    return False

In [77]:
# Calculate data type accuracy

def calculate_dtype_accuracy(
    dataframe,
    expected_categories
):

    correct_columns = 0
    checked_columns = 0

    for column, expected_category in expected_categories.items():

        if column in dataframe.columns:

            checked_columns += 1

            if check_dtype_category(
                dataframe[column],
                expected_category
            ):
                correct_columns += 1

    return round(
        (correct_columns / checked_columns) * 100,
        2
    )

In [78]:
# Calculate before and after data type accuracy

dtype_accuracy_before = (
    calculate_dtype_accuracy(
        df_original,
        expected_type_categories
    )
)

dtype_accuracy_after = (
    calculate_dtype_accuracy(
        df,
        expected_type_categories
    )
)

print(
    f"Dtype Accuracy Before Cleaning: "
    f"{dtype_accuracy_before}%"
)

print(
    f"Dtype Accuracy After Cleaning: "
    f"{dtype_accuracy_after}%"
)

Dtype Accuracy Before Cleaning: 75.0%
Dtype Accuracy After Cleaning: 100.0%


## BEFORE VS AFTER SUMMARY

In [79]:
# Create Final Before vs After Table

before_after_summary = pd.DataFrame({

    "Metric": [
        "Row Count",
        "Column Count",
        "Total Missing Values",
        "Duplicate Rows",
        "Dtype Accuracy (%)"
    ],

    "Before Cleaning": [
        len(df_original),
        len(df_original.columns),
        df_original.isna().sum().sum(),
        df_original.duplicated().sum(),
        dtype_accuracy_before
    ],

    "After Cleaning": [
        len(df),
        len(df.columns),
        df.isna().sum().sum(),
        df.duplicated().sum(),
        dtype_accuracy_after
    ]
})

before_after_summary

,Metric,Before Cleaning,After Cleaning
0,Row Count,1020.0,1020.0
1,Column Count,12.0,14.0
2,Total Missing Values,235.0,0.0
3,Duplicate Rows,0.0,0.0
4,Dtype Accuracy (%),75.0,100.0


In [80]:
# Column-Level Missing Value Comparison

common_columns = [
    column
    for column in df_original.columns
    if column in df.columns
]

null_comparison = pd.DataFrame({

    "Column": common_columns,

    "Nulls Before": [
        df_original[column].isna().sum()
        for column in common_columns
    ],

    "Nulls After": [
        df[column].isna().sum()
        for column in common_columns
    ]
})

null_comparison

,Column,Nulls Before,Nulls After
0,Employee_ID,0,0
1,First_Name,0,0
2,Last_Name,0,0
3,Age,211,0
4,Department_Region,0,0
5,Status,0,0
6,Join_Date,0,0
7,Salary,24,0
8,Email,0,0
9,Phone,0,0


## FINAL CLEAN DATASET PREVIEW

In [81]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work,Department,Region
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,2021-04-02,59767.65,bob.davis@example.com,1651623197,Average,True,Devops,California
1,EMP1001,Bob,Brown,30.0,Finance-Texas,Active,2020-07-10,65304.66,bob.brown@example.com,1898471390,Excellent,True,Finance,Texas
2,EMP1002,Alice,Jones,30.0,Admin-Nevada,Pending,2023-12-07,88145.90,alice.jones@example.com,5596363211,Good,True,Admin,Nevada
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,2021-11-27,69450.99,eva.davis@example.com,3476490784,Good,True,Admin,Nevada
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,2022-01-05,109324.61,frank.williams@example.com,1586734256,Poor,False,Cloud Tech,Florida


In [82]:
# Display Random Sample
df.sample(
    min(10, len(df)),
    random_state=42
)

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work,Department,Region
523,EMP1523,Alice,Davis,25.0,DevOps-Texas,Inactive,2022-06-28,76617.76,alice.davis@example.com,7314976460,Excellent,True,Devops,Texas
602,EMP1602,Charlie,Brown,30.0,Admin-Florida,Active,2021-01-01,89645.01,charlie.brown@example.com,9596604872,Good,True,Admin,Florida
526,EMP1526,Bob,Williams,30.0,DevOps-Illinois,Active,2021-08-01,57980.88,bob.williams@example.com,6375791451,Poor,False,Devops,Illinois
31,EMP1031,Bob,Smith,25.0,DevOps-Illinois,Active,2020-07-10,114845.69,bob.smith@example.com,6591520176,Excellent,True,Devops,Illinois
616,EMP1616,Frank,Smith,40.0,Sales-Illinois,Active,2021-04-15,76474.79,frank.smith@example.com,6876112490,Excellent,True,Sales,Illinois
585,EMP1585,Heidi,Davis,30.0,Sales-New York,Active,2024-03-09,77369.48,heidi.davis@example.com,2059438975,Average,False,Sales,New York
444,EMP1444,Grace,Brown,30.0,DevOps-Illinois,Active,2023-07-15,119586.11,grace.brown@example.com,9815287338,Average,True,Devops,Illinois
732,EMP1732,Charlie,Smith,40.0,Cloud Tech-New York,Active,2023-11-04,76167.46,charlie.smith@example.com,7282709080,Good,False,Cloud Tech,New York
76,EMP1076,David,Brown,40.0,Admin-California,Inactive,2022-09-21,52031.74,david.brown@example.com,1589357125,Good,True,Admin,California
411,EMP1411,Grace,Garcia,25.0,DevOps-Illinois,Pending,2023-06-15,60687.63,grace.garcia@example.com,9859167953,Excellent,False,Devops,Illinois


In [83]:
# Save Dataset to New CSV
output_file = (
    "Cleaned_Employee_Dataset.csv"
)

df.to_csv(
    output_file,
    index=False
)

print(
    f"Cleaned dataset saved successfully as: "
    f"{output_file}"
)

Cleaned dataset saved successfully as: Cleaned_Employee_Dataset.csv


In [84]:
# Verify File Creation
import os

if os.path.exists(output_file):

    print(
        "File saved successfully."
    )

    print(
        f"File size: "
        f"{os.path.getsize(output_file)} bytes"
    )

else:

    print(
        "File was not found."
    )

File saved successfully.
File size: 134731 bytes


# Final Conclusion

This project successfully transformed the original messy employee dataset into a clean and analysis-ready dataset through a systematic data cleaning process.

The initial data quality assessment identified 235 missing values, data type inconsistencies and formatting issues across several variables. Missing values in the `Age` and `Salary` columns were handled using median imputation because the median provides a robust estimate of central tendency and is less sensitive to extreme values.

Duplicate records were examined before cleaning. No exact duplicate rows were identified; therefore, no records required removal. Text-based variables were standardised by removing unnecessary spaces and applying consistent formatting. Names, categorical variables and email addresses were also normalised to improve consistency.

The `Join_Date` column was converted to a datetime format, while identifier variables such as `Employee_ID` and `Phone` were stored as strings. The `Department_Region` variable was additionally separated into individual `Department` and `Region` columns to improve the usability of the dataset for future analysis.

Outlier detection was conducted using the Interquartile Range (IQR) method for the numerical variables `Age` and `Salary`. No outliers were identified; therefore, no records required removal or modification.

Following the cleaning process, the final dataset contained 1,020 records, zero missing values and zero duplicate rows. The cleaned dataset was saved as a new CSV file and is suitable for further exploratory analysis, statistical analysis and data visualisation.